In [1]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from dask.distributed import Client, LocalCluster

# ============================
# User settings
# ============================

# Generator site list
gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")

boco_ds = "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/C2_boco_wind_vec.nc"
clust_ds = "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/C2_cluster_wind_vec.nc"

output_dir = '/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/hourly_composites'

In [2]:
client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 4
Total threads: 4,Total memory: 18.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34195,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:44317,Total threads: 1
Dashboard: /proxy/43127/status,Memory: 4.50 GiB
Nanny: tcp://127.0.0.1:42609,


In [3]:
cluster = ['TARALGA1','CROOKWF2',
            'GULLRWF1',
            'GULLRWF2',
            'GUNNING1',
            'BANGOWF1',
            'BANGOWF2',
            'COLWF01',
            'WOODLWN1',
            'BOCORWF1']

cluster = gen_csv[gen_csv['DUID'].isin(cluster)][['DUID','lat','lon']]

In [4]:
ds = xr.open_dataset("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/hourly_uv_composite.nc",
                   engine='netcdf4',
                   chunks='auto')
ds

<xarray.Dataset> Size: 4MB
Dimensions:  (hour: 24, lat: 125, lon: 88)
Coordinates:
    height   float64 8B ...
  * lon      (lon) float64 704B 147.5 147.5 147.6 147.6 ... 150.9 150.9 151.0
  * lat      (lat) float64 1kB -38.49 -38.45 -38.41 ... -33.61 -33.57 -33.53
    crs      int32 4B ...
  * hour     (hour) int64 192B -9223372036854775806 ... -9223372036854775806
Data variables:
    ua100m   (hour, lat, lon) float64 2MB dask.array<chunksize=(24, 125, 88), meta=np.ndarray>
    va100m   (hour, lat, lon) float64 2MB dask.array<chunksize=(24, 125, 88), meta=np.ndarray>
Attributes: (12/60)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1H.json
    productive_version:        462b27f
    variable_version:          v20240809
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    date_modified:             2024-10-11T00:33:24Z
    date_metadata_modified:    2024-10-11T00:33:24Z
    history:                   Tue Aug 06 06:47:11 2024: /g/data/access/ngm/m...
    references:                https://doi.org/10.25914/1x6g-2v48
    license:                   https://doi.org/10.25914/1x6g-2v48
    acknowledgement:           The production of BARRA2 was supported with fu...

In [5]:
def plot_frame(u, v, lat, lon, t, output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/',
               quiver_scale=400, extent=[147.5, 151, -38.5, -33.5], cluster=cluster, highlight_id='BOCORWF1'):
    """
    Plot wind vectors with optional cluster points.
    
    cluster: DataFrame with columns ['ID', 'lat', 'lon']
    highlight_id: specific ID to highlight in red
    """
    speed = np.sqrt(u**2 + v**2)
    plt.figure(figsize=(10,8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, lw=1.5)
    ax.add_feature(cfeature.BORDERS, linestyle='-', lw=1.5)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.5)
    
    # Contour of wind speed
    plt.contourf(lon, lat, speed, cmap='cividis', transform=ccrs.PlateCarree())
    plt.colorbar(label='Wind speed (m/s)')
    
    # Quiver vectors
    plt.quiver(lon, lat, u, v, scale=quiver_scale, color='white', transform=ccrs.PlateCarree())
    
    # Plot cluster points
    if cluster is not None:
        # All points in orange
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", alpha=0.6, s=50, label="Wind Farms", zorder=5, transform=ccrs.PlateCarree())
        
        # Highlight one point in red
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'], 
                       color="red", alpha=0.6, s=80, label=f"ID {highlight_id}", zorder=6,
                       transform=ccrs.PlateCarree())

    plt.title(f'Wind vectors at {str(t)}')
    
    filename = make_filename(t, output_dir)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    return filename


def make_filename(t, output_dir, prefix="wind"):
    try:
        # If t is datetime-like, use date formatting
        dt_str = np.datetime_as_string(t, unit='m')
        dt_str = dt_str.replace('-', '')[2:8] + '_' + dt_str[11:13] + dt_str[14:16]
    except Exception:
        # If it's just an index/hour, format as hour
        if isinstance(t, (int, np.integer)):
            dt_str = f"hour{t:02d}"
        else:
            dt_str = str(t).replace(":", "").replace(" ", "_")
    
    return os.path.join(output_dir, f"{prefix}_{dt_str}.png")


In [30]:
extent = [147.5, 151, -38.5, -33.5]
step = 2


for i in range(len(ds.hour)):
        u = ds['ua100m'].isel(hour=i)[::step, ::step]
        v = ds['va100m'].isel(hour=i)[::step, ::step]
        lat = ds['lat'][::2].values
        lon = ds['lon'][::step].values
        t = ds.hour[i].values
        
        plot_frame(u, v, lat, lon, t, output_dir, 300, extent)

In [31]:
ds

<xarray.Dataset> Size: 4MB
Dimensions:  (hour: 24, lat: 125, lon: 88)
Coordinates:
    height   float64 8B ...
  * lon      (lon) float64 704B 147.5 147.5 147.6 147.6 ... 150.9 150.9 151.0
  * lat      (lat) float64 1kB -38.49 -38.45 -38.41 ... -33.61 -33.57 -33.53
    crs      int32 4B ...
  * hour     (hour) int64 192B 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
Data variables:
    ua100m   (hour, lat, lon) float64 2MB dask.array<chunksize=(24, 125, 88), meta=np.ndarray>
    va100m   (hour, lat, lon) float64 2MB dask.array<chunksize=(24, 125, 88), meta=np.ndarray>
Attributes: (12/60)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1H.json
    productive_version:        462b27f
    variable_version:          v20240809
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    date_modified:             2024-10-11T00:33:24Z
    date_metadata_modified:    2024-10-11T00:33:24Z
    history:                   Tue Aug 06 06:47:11 2024: /g/data/access/ngm/m...
    references:                https://doi.org/10.25914/1x6g-2v48
    license:                   https://doi.org/10.25914/1x6g-2v48
    acknowledgement:           The production of BARRA2 was supported with fu...

In [8]:
# Lazy hourly mean
hourly_composite =  ds_subset.groupby("time.hour").mean()

# Compute in parallel
with ProgressBar():
    hourly_composite = hourly_composite.compute()